# Preprocessing - life_style_data.duckdb

## Importation des librairies essentielles

In [37]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import OrdinalEncoder

## Nettoyage des données

### Contexte & objectifs

Ce chapitre prépare le dataset life_style_data pour la modélisation.
Objectifs :
- Corriger les types (notamment les décimales avec virgules) ;
- Quantifier et traiter les valeurs manquantes ;
- Détecter/supprimer les doublons ;
- Poser des règles de plausibilité métier (sans censurer) ;
- Produire un rapport de nettoyage et des artefacts (dataset nettoyé).

Entrée : table DuckDB life_style_data (chargée via DBeaver).
Sorties : df_clean (Parquet/CSV), tableaux de suivi (manquants, conversions, doublons, flags).

### Imports, paramètres, chargement depuis DuckDB

In [38]:
# 🔧 Définition des chemins (adaptés à ton arborescence GitHub)
# Notebook : C:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\notebooks\preprocessing\life_style_data
# Base     : C:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\data\raw\life_style_data

# Le notebook s’exécute depuis son répertoire → on peut repartir du cwd
current_dir = Path.cwd()
project_root = current_dir.parents[3]  # remonte jusqu’à "train.me"

# 🗂️ Chemins complets
db_path = project_root / "src" / "data" / "raw" / "life_style_data" / "life_style_data"
export_dir = project_root / "src" / "data" / "processed" / "life_style_data"

# ✅ Connexion à la base DuckDB
con = duckdb.connect(str(db_path))

# ✅ Chargement de la table
df = con.execute("SELECT * FROM life_style_data").fetchdf()

# ✅ Vérification du chargement
shape_before = df.shape
print(f"✔ Données chargées : {shape_before[0]} lignes, {shape_before[1]} colonnes")
display(df.head(3))

✔ Données chargées : 20000 lignes, 54 colonnes


,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,...,cal_from_macros,pct_carbs,protein_per_kg,pct_HRR,pct_maxHR,cal_balance,lean_mass_kg,expected_burn,Burns Calories (per 30 min)_bc,Burns_Calories_Bin
0,34.91,Male,65.27,1.62,188.58,157.65,69.05,1.00,1080.90,Strength,...,2139.59,0.500432,1.624789,0.741237,0.835985,725.10,47.777394,685.1600,7.260425e+19,Medium
1,23.37,Female,56.41,1.55,179.43,131.75,73.18,1.37,1809.91,HIIT,...,1711.65,0.500850,1.514093,0.551247,0.734270,-232.91,40.809803,978.6184,1.020506e+20,High
2,33.20,Female,58.98,1.67,175.04,123.95,54.96,0.91,802.26,Cardio,...,1965.92,0.500610,1.663445,0.574534,0.708124,805.74,44.635580,654.5266,1.079607e+20,High


### Audit qualité initial

On dresse un état des lieux :
- Répartition des types et cardinalités ;
- Valeurs manquantes (%) par colonne ;
- Doublons potentiels.

Cet instantané nous sert de référence avant toute modification.

In [39]:
# 1.2 Audit qualité initial
missing_pct = df.isna().mean().sort_values(ascending=False) * 100
dup_count = df.duplicated().sum()

summary_types = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "missing_%": df.isna().mean() * 100
}).sort_values("missing_%", ascending=False)

print(f"• Doublons détectés (avant): {dup_count}")
display(missing_pct.to_frame("missing_%").head(20))
display(summary_types.head(20))


• Doublons détectés (avant): 0


,missing_%
Age,0.0
Body Part,0.0
prep_time_min,0.0
cook_time_min,0.0
rating,0.0
Name of Exercise,0.0
Sets,0.0
Reps,0.0
Benefit,0.0
Burns Calories (per 30 min),0.0


,dtype,n_unique,missing_%
Age,float64,3950,0.0
Body Part,object,7,0.0
prep_time_min,float64,5384,0.0
cook_time_min,float64,9406,0.0
rating,float64,414,0.0
Name of Exercise,object,55,0.0
Sets,float64,30,0.0
Reps,float64,654,0.0
Benefit,object,49,0.0
Burns Calories (per 30 min),float64,5766,0.0


### Correction de types (décimales à virgule, espaces insécables)

Dans de nombreux exports, les colonnes numériques arrivent en texte (object) avec :
- des virgules à la place du point décimal ;
- des espaces insécables (\u00A0) ou espaces.

On corrige proprement pour garantir la cohérence numérique.

In [40]:
# 1.3 Conversion robuste des colonnes numériques candidates (si présentes)
num_candidates = [
    "Age","Weight (kg)","Height (m)","Max_BPM","Avg_BPM","Resting_BPM",
    "Session_Duration (hours)","Fat_Percentage","Water_Intake (liters)",
    "Workout_Frequency (days/week)","BMI","Carbs","Proteins","Fats","Calories",
    "sugar_g","sodium_mg","cholesterol_mg","serving_size_g","prep_time_min","cook_time_min",
    "rating","BMI_calc","cal_from_macros","pct_carbs","protein_per_kg","pct_HRR","pct_maxHR",
    "cal_balance","lean_mass_kg","expected_burn","Burns Calories (per 30 min)","Calories_Burned"
]

convert_log = []
df_conv = df.copy()

for c in [col for col in num_candidates if col in df_conv.columns]:
    if df_conv[c].dtype == "object":
        before_na = df_conv[c].isna().sum()
        df_conv[c] = (df_conv[c]
            .astype(str)
            .str.replace(",", ".", regex=False)     # virgule ⇒ point
            .str.replace("\u00A0", "", regex=False) # espace insécable
            .str.strip()
        )
        df_conv[c] = pd.to_numeric(df_conv[c], errors="coerce")
        after_na = df_conv[c].isna().sum()
        convert_log.append({"column": c, "was_object": True, "na_before": before_na, "na_after": after_na})

convert_report = pd.DataFrame(convert_log).sort_values("column") if convert_log else pd.DataFrame(columns=["column","was_object","na_before","na_after"])
display(convert_report)


,column,was_object,na_before,na_after


### Règles de plausibilité (flags non bloquants)

On applique des bornes métier pour repérer des valeurs possiblement aberrantes.
On ne supprime pas : on flague pour informer la modélisation et documenter les décisions.

Exemples (paramétrables) :
- Age ∈ [10, 90]
- Max_BPM ∈ [80, 230]
- Session_Duration (hours) ∈ [0.1, 6]
- BMI ∈ [10, 60]

In [41]:
# 1.4 Flags de plausibilité (ne supprime rien)
df_flag = df_conv.copy()

def safe_between(s, low, high):
    return s.between(low, high) if s.notna().any() else pd.Series(False, index=s.index)

flags = {}
if "Age" in df_flag:               flags["Age_out"] = ~safe_between(df_flag["Age"], 10, 90)
if "Max_BPM" in df_flag:           flags["BPM_out"] = ~safe_between(df_flag["Max_BPM"], 80, 230)
if "Session_Duration (hours)" in df_flag: flags["SessDur_out"] = ~safe_between(df_flag["Session_Duration (hours)"], 0.1, 6)
if "BMI" in df_flag:               flags["BMI_out"] = ~safe_between(df_flag["BMI"], 10, 60)

flag_counts = {k: v.fillna(False).sum() for k, v in flags.items()}
flag_table = pd.DataFrame.from_dict(flag_counts, orient="index", columns=["nb_outliers"]).sort_index()
display(flag_table)


,nb_outliers
Age_out,0
BMI_out,0
BPM_out,0
SessDur_out,0


### Imputation des valeurs manquantes (baseline)

Stratégie simple et robuste (baseline reproductible) :
- Numériques → médiane (moins sensible aux outliers) ;
- Catégorielles → mode (valeur la plus fréquente, sinon “Unknown”).

Cette baseline est suffisante pour itérer ; on pourra spécialiser par la suite.

In [42]:
# 1.5 Imputation des manquants
df_imp = df_flag.copy()

num_cols = df_imp.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df_imp.columns if c not in num_cols]

# Numériques → médiane
for c in num_cols:
    if df_imp[c].isna().any():
        df_imp[c] = df_imp[c].fillna(df_imp[c].median())

# Catégorielles → mode (ou "Unknown")
for c in cat_cols:
    if df_imp[c].isna().any():
        m = df_imp[c].mode(dropna=True)
        df_imp[c] = df_imp[c].fillna(m.iloc[0] if not m.empty else "Unknown")

missing_before = missing_pct  # gardé depuis 1.2
missing_after = df_imp.isna().mean().sort_values(ascending=False) * 100

print("✔ Imputation effectuée.")
display(pd.DataFrame({"missing_before_%": missing_before}).head(10))
display(pd.DataFrame({"missing_after_%": missing_after}).head(10))


✔ Imputation effectuée.


,missing_before_%
Age,0.0
Body Part,0.0
prep_time_min,0.0
cook_time_min,0.0
rating,0.0
Name of Exercise,0.0
Sets,0.0
Reps,0.0
Benefit,0.0
Burns Calories (per 30 min),0.0


,missing_after_%
Age,0.0
Body Part,0.0
prep_time_min,0.0
cook_time_min,0.0
rating,0.0
Name of Exercise,0.0
Sets,0.0
Reps,0.0
Benefit,0.0
Burns Calories (per 30 min),0.0


### Doublons & incohérences simples

On élimine les doublons stricts (lignes identiques).

Cette opération stabilise la taille du corpus et évite des biais à l’entraînement.

In [43]:
# 1.6 Doublons
rows_before = df_imp.shape[0]
dup_total = df_imp.duplicated().sum()
df_clean = df_imp.drop_duplicates().reset_index(drop=True)
rows_after = df_clean.shape[0]

clean_report = pd.DataFrame({
    "rows_before":[rows_before],
    "rows_after":[rows_after],
    "duplicates_removed":[dup_total],
    "cols":[df_clean.shape[1]]
})
display(clean_report)


,rows_before,rows_after,duplicates_removed,cols
0,20000,20000,0,54


### Rapport de nettoyage (synthèse consultant)

Nous consolidons les éléments clés pour traçabilité et communication :
- Colonnes converties object → numeric ;
- Avant/Après des valeurs manquantes ;
- Nombre de doublons supprimés ;
- Flags de plausibilité relevés.

Ce rapport sert de référence pour justifier les choix et préparer la suite (encodage, scaling).

In [44]:
# 1.7 Rapport consolidé
sections = {
    "convert_report": convert_report,
    "missing_before_% (top 20)": missing_before.to_frame("missing_%").head(20),
    "missing_after_% (top 20)": missing_after.to_frame("missing_%").head(20),
    "plausibility_flags": flag_table,
    "clean_report": clean_report
}

for title, df_sec in sections.items():
    print("\n" + "="*30 + f" {title} " + "="*30)
    display(df_sec)



============================== convert_report ==============================


,column,was_object,na_before,na_after



============================== missing_before_% (top 20) ==============================


,missing_%
Age,0.0
Body Part,0.0
prep_time_min,0.0
cook_time_min,0.0
rating,0.0
Name of Exercise,0.0
Sets,0.0
Reps,0.0
Benefit,0.0
Burns Calories (per 30 min),0.0



============================== missing_after_% (top 20) ==============================


,missing_%
Age,0.0
Body Part,0.0
prep_time_min,0.0
cook_time_min,0.0
rating,0.0
Name of Exercise,0.0
Sets,0.0
Reps,0.0
Benefit,0.0
Burns Calories (per 30 min),0.0



============================== plausibility_flags ==============================


,nb_outliers
Age_out,0
BMI_out,0
BPM_out,0
SessDur_out,0



============================== clean_report ==============================


,rows_before,rows_after,duplicates_removed,cols
0,20000,20000,0,54


### Export des artefacts (dataset nettoyé)

On persiste le dataset nettoyé pour les étapes suivantes (encodage, scaling, split) et pour reproductibilité :
- life_style_data_clean.parquet (format recommandé) ;
- Optionnel : life_style_data_clean.csv.

In [45]:
# 1.8 Export artefacts
parquet_path = export_dir / "life_style_data_clean.parquet"
csv_path     = export_dir / "life_style_data_clean.csv"

df_clean.to_parquet(parquet_path, index=False, engine="fastparquet")
df_clean.to_csv(csv_path, index=False)

print(f"✔ Export parquet : {parquet_path}")
print(f"✔ Export csv     : {csv_path}")
print(f"✔ Shape final    : {df_clean.shape}")


✔ Export parquet : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\data\processed\life_style_data\life_style_data_clean.parquet
✔ Export csv     : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\data\processed\life_style_data\life_style_data_clean.csv
✔ Shape final    : (20000, 54)


### Critères d’acceptation & décisions

- Types numériques/catégoriels conformes au dictionnaire ;
- Manquants résiduels documentés et ≤ seuils tolérés (par défaut, 5%) ;
- Doublons supprimés (ou chiffrés et justifiés si conservés) ;
- Flags de plausibilité suivis (pas de censure sans décision métier explicite).

Décisions à garder au journal :
- Colonnes converties, colonnes fortement manquantes et stratégie appliquée ;
- Seuils de plausibilité retenus et impact (aucune ligne supprimée à ce stade) ;
- Lieux d’export des artefacts.

## Encodage des variables catégorielles

## Objectifs

Cette étape consiste à transformer les variables catégorielles du jeu de données nettoyé en représentations numériques exploitables par les algorithmes de Machine Learning.
L’encodage rend les colonnes qualitatives compatibles avec les modèles tout en préservant l’information sémantique.
Deux approches principales sont appliquées selon la nature des catégories :
- Encodage ordinal pour les variables hiérarchiques (ex. niveau d’expérience) ;
- Encodage one-hot pour les variables nominales (ex. type d’exercice, partie du corps, équipement).

### Chargement du dataset nettoyé

In [46]:
# Charger les artefacts nettoyés
df_clean = pd.read_parquet(parquet_path)

print(f"✔ Dataset chargé : {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes")
df_clean.head(3)

✔ Dataset chargé : 20000 lignes, 54 colonnes


,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,...,cal_from_macros,pct_carbs,protein_per_kg,pct_HRR,pct_maxHR,cal_balance,lean_mass_kg,expected_burn,Burns Calories (per 30 min)_bc,Burns_Calories_Bin
0,34.91,Male,65.27,1.62,188.58,157.65,69.05,1.00,1080.90,Strength,...,2139.59,0.500432,1.624789,0.741237,0.835985,725.10,47.777394,685.1600,7.260425e+19,Medium
1,23.37,Female,56.41,1.55,179.43,131.75,73.18,1.37,1809.91,HIIT,...,1711.65,0.500850,1.514093,0.551247,0.734270,-232.91,40.809803,978.6184,1.020506e+20,High
2,33.20,Female,58.98,1.67,175.04,123.95,54.96,0.91,802.26,Cardio,...,1965.92,0.500610,1.663445,0.574534,0.708124,805.74,44.635580,654.5266,1.079607e+20,High


### Identification des variables catégorielles
On repère les variables non numériques à encoder.
L’objectif est de distinguer :
- les variables nominales (non ordonnées),
- les variables ordinales (possédant une hiérarchie logique).

In [47]:
# Identification des colonnes catégorielles
cat_cols = df_clean.select_dtypes(exclude=["number"]).columns.tolist()
print(f"🔎 {len(cat_cols)} variables catégorielles détectées :")
cat_cols


🔎 15 variables catégorielles détectées :


['Gender',
 'Workout_Type',
 'meal_name',
 'meal_type',
 'diet_type',
 'cooking_method',
 'Name of Exercise',
 'Benefit',
 'Target Muscle Group',
 'Equipment Needed',
 'Difficulty Level',
 'Body Part',
 'Type of Muscle',
 'Workout',
 'Burns_Calories_Bin']

### Encodage ordinal

Les variables catégorielles ordonnées (par ex. Experience_Level, Difficulty Level) sont converties en valeurs entières selon leur progression logique.

Cette méthode conserve la hiérarchie implicite entre les modalités.

In [48]:
# Encodage ordinal sur variables à hiérarchie logique
ordinal_features = ["Experience_Level", "Difficulty Level"]

encoder_ordinal = OrdinalEncoder()
df_clean[ordinal_features] = encoder_ordinal.fit_transform(df_clean[ordinal_features])

print("✔ Encodage ordinal appliqué sur :", ordinal_features)
df_clean[ordinal_features].head()


✔ Encodage ordinal appliqué sur : ['Experience_Level', 'Difficulty Level']


,Experience_Level,Difficulty Level
0,14.0,0.0
1,14.0,2.0
2,2.0,2.0
3,12.0,0.0
4,13.0,0.0


### Encodage One-Hot

Les variables nominales (sans hiérarchie) sont transformées par One-Hot Encoding, créant une colonne binaire par modalité.

Cela permet d’éviter toute interprétation de relation d’ordre entre catégories.

In [49]:
df_encoded = pd.get_dummies(
    df_clean,
    columns=[col for col in cat_cols if col not in ["Experience_Level", "Difficulty Level"]],
    drop_first=True
)

print(f"✔ One-Hot Encoding appliqué : {df_encoded.shape[1]} colonnes au total")
df_encoded.head(3)


✔ One-Hot Encoding appliqué : 287 colonnes au total


,Age,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Fat_Percentage,Water_Intake (liters),...,Workout_Skull crushers,Workout_Standing calf raises,Workout_Towel pull-up,Workout_Triceps dips,Workout_Triceps pushdowns,Workout_Wrist curl,Workout_Wrist extension,Burns_Calories_Bin_Low,Burns_Calories_Bin_Medium,Burns_Calories_Bin_Very High
0,34.91,65.27,1.62,188.58,157.65,69.05,1.00,1080.90,26.800377,1.50,...,False,False,False,False,False,False,False,False,True,False
1,23.37,56.41,1.55,179.43,131.75,73.18,1.37,1809.91,27.655021,1.90,...,False,False,False,False,False,False,False,False,False,False
2,33.20,58.98,1.67,175.04,123.95,54.96,0.91,802.26,24.320821,1.88,...,False,True,False,False,False,False,False,False,False,False


### Vérification post-encodage

Après encodage :
- Toutes les variables sont numériques ;
- Le nombre de colonnes augmente (selon le nombre de modalités) ;
- Aucun NaN ne doit être réintroduit.

In [50]:
print(f"Forme finale du dataset encodé : {df_encoded.shape}")
print(f"Valeurs manquantes totales : {df_encoded.isna().sum().sum()}")

# Aperçu d'un sous-ensemble des nouvelles colonnes encodées
df_encoded.filter(like="Workout_Type").head(5)


Forme finale du dataset encodé : (20000, 287)
Valeurs manquantes totales : 0


,Workout_Type_HIIT,Workout_Type_Strength,Workout_Type_Yoga
0,False,True,False
1,True,False,False
2,False,False,False
3,True,False,False
4,False,True,False


### Export du dataset encodé

On exporte le jeu de données final encodé pour les étapes suivantes :
- Mise à l’échelle / normalisation
- Split train/test

In [51]:
encoded_path = export_dir / "life_style_data_encoded.parquet"
df_encoded.to_parquet(encoded_path, index=False)
print(f"✔ Export du dataset encodé : {encoded_path}")


✔ Export du dataset encodé : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\data\processed\life_style_data\life_style_data_encoded.parquet


L’encodage garantit la compatibilité du jeu de données avec les algorithmes de Machine Learning.

Les variables catégorielles hiérarchiques ont été traitées par encodage ordinal, et les variables nominales par encodage one-hot, réduisant les risques de biais d’interprétation.

# Code complet avec pipeline

In [52]:
# pipeline = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler', StandardScaler()),
#     ('model', LinearRegression())
# ])

In [53]:
# pipeline.fit(X_train, y_train)

In [54]:
# y_pred = pipeline.predict(X_test)
# pd.DataFrame(y_pred,columns=['Predicted_PRICE'])